In [1]:
import streamlit as st
import pandas as pd
import joblib
import plotly.express as px
import requests

st.set_page_config(
    page_title = "NHS_Diagnostic Breach Risk",
    layout = "wide"
)

@st.cache_data
def load_data():
    df = pd.read_csv('predictions_dec2025.csv')
    return df

@st.cache_data
def load_geojson():
    url = "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Integrated_Care_Boards_April_2023_EN_BGC/FeatureServer/0/query?where=1%3D1&outFields=*&f=geojson"
    response = requests.get(url)
    return response.json()

@st.cache_resource 
def load_model():
    model = joblib.load('xgb_model.pkl')
    scaler = joblib.load('scaler.pkl')
    return model, scaler

df = load_data()
model, scaler = load_model()
geojson = load_geojson()

st.title("NHS Diagnostic Breach Risk Dashboard")
st.markdown("Predicted breach risk for March 2026 based on December 2025 diagnostic data")




2026-03-13 19:25:49.063 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 19:25:49.064 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-13 19:25:49.065 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-13 19:25:49.066 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-13 19:25:49.066 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 19:25:49.132 
  command:

    streamlit run /opt/anaconda3/lib/python3.13/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-03-13 19:25:49.133 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running i

DeltaGenerator()

In [ ]:
## choropleth map:

## Mapping chloropeth
## get ONS code matched with the name, add to icb_risk 

icb_risk = df.groupby('provider_parent_name').agg(
    avg_breach_prob = ('breach_risk_prob', 'mean')
).reset_index()

icb_risk['name_upper'] = icb_risk['provider_parent_name'].str.strip().str.upper()

geojson_name_to_code = {
    f['properties']['ICB23NM'].upper().strip(): f['properties']['ICB23CD']
    for f in geojson['features']
}

icb_risk['icb_ons_code'] = icb_risk['name_upper'].map(geojson_name_to_code)

# missing ICB data
missing_icbs = [
    { 
        'provider_parent_name': f['properties']['ICB23NM'],
        'avg_breach_prob': None,
        'icb_ons_code': f['properties']['ICB23CD'],
        'name_upper': f['properties']['ICB23NM'].upper()
    }
    for f in geojson['features']
    if f['properties']['ICB23CD'] not in icb_risk['icb_ons_code'].values 
]

missing_df = pd.DataFrame(missing_icbs)

icb_risk_full = pd.concat([icb_risk, missing_df], ignore_index = True)

fig = px.choropleth(
    icb_risk_full,
    geojson=geojson,
    locations = 'icb_ons_code',
    featureidkey = 'properties.ICB23CD',
    color = 'avg_breach_prob',
    color_continuous_scale = ['green', 'yellow', 'red'],
    hover_name = 'provider_parent_name',
    hover_data= {
        'avg_breach_prob': 'Avg Breach Probability',
        'icb_ons_code': False,
        'name_upper': False
    },
    labels={
        'avg_breach_prob': 'Avg Breach Probability'
    },
    title = 'NHS ICB Diagnostic Breach Risk - March 2026'
)
